# Kaggle Training: LogFilter BiEncoder on log ↔ ATT&CK pairs

Fine-tunes `cisco-ai/SecureBERT2.0-biencoder` as a sentence-transformers BiEncoder for two production jobs: near-duplicate log detection and ATT&CK candidate retrieval. `LogScorer` embeds each log, checks it against a rolling dedup index, retrieves top-k ATT&CK techniques, and then sends those candidates to the CrossEncoder.

Bootstrap signal comes from the same high-precision `KEYWORD_MAP` used by the CrossEncoder notebook. Each keyword hit creates a positive `(log_window, technique_text)` pair. Training uses `MultipleNegativesRankingLoss`, so other positives in the batch become in-batch negatives without needing hand-labelled negative pairs.

Output: `models/biencoder/final/` - a native sentence-transformers directory drop-in for `src/logfilter/models/biencoder.py`. Expected runtime on Kaggle Free GPU (T4) is well under the 9h session limit for the sampled run. Enable a GPU accelerator before running training cells.

## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)

## 2. Install training dependencies

Adds `sentence-transformers` for the BiEncoder training loop on top of the standard transformers/datasets stack. The small environment cell keeps Kaggle on one visible GPU, prints `KAGGLE_KERNEL_RUN_TYPE`, and disables compile paths that can be fragile on P100 while still working on T4.

In [ ]:
%pip install -q transformers datasets accelerate sentence-transformers scikit-learn

In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')
print('Kaggle run type:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'local-or-interactive'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

## 3. Attach the HDFS TraceBench dataset and load MITRE techniques

The training data is built from two existing repo artifacts: reconstructed text windows from HDFS TraceBench and `config/mitre_techniques.json` technique descriptions, matching the retrieval corpus indexed by the production BiEncoder wrapper.

In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
MITRE_PATH = REPO_DIR / 'config' / 'mitre_techniques.json'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
    if not candidates:
        raise FileNotFoundError('No Kaggle input folder with normal_trace.csv and failure_trace.csv was found.')
    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset available:', PREPROCESSED_DIR)

if not MITRE_PATH.exists():
    raise FileNotFoundError(f'Missing MITRE technique seed: {MITRE_PATH}')
techniques = json.loads(MITRE_PATH.read_text())
tech_by_id = {t['id']: t for t in techniques}
print(f'Loaded {len(techniques)} ATT&CK techniques')

## 4. Build positive pairs from keyword heuristics

`KEYWORD_MAP` is the shared hand-crafted asset from the CrossEncoder notebook. Each high-precision keyword hit creates a positive `(log_window, technique_text)` pair for BiEncoder contrastive training. Texts with zero keyword hits are dropped from the labelled-pair path.

Because `MultipleNegativesRankingLoss` treats other batch positives as negatives, this section intentionally builds only positive pairs. Keep `KEYWORD_MAP` conservative: a noisy positive pair is worse than a missing one.

In [ ]:
import random
from training.text_dataset import build_windows

# Keyword → ATT&CK technique IDs. Keep keys lowercase; matching is case-insensitive.
# Extend conservatively when adding new log sources; high precision > high recall here.
KEYWORD_MAP: dict[str, list[str]] = {
    # Auth / credentials
    'permission denied': ['T1078'],
    'access denied': ['T1078'],
    'accesscontrolexception': ['T1078'],
    'authentication failed': ['T1110'],
    'authorization failed': ['T1078'],
    'invalid credentials': ['T1110'],
    'login failed': ['T1110'],
    'kerberos': ['T1003'],
    'credential': ['T1003'],
    # Remote services / lateral
    'ssh': ['T1021', 'T1021.004'],
    'rdp': ['T1021.001'],
    'smb://': ['T1021.002'],
    '\\': ['T1021.002'],          # UNC path
    'winrm': ['T1021.006'],
    'telnet': ['T1021'],
    # Execution / scripting
    'powershell': ['T1059.001'],
    'cmd.exe': ['T1059.003'],
    '/bin/sh': ['T1059.004'],
    '/bin/bash': ['T1059.004'],
    'wscript': ['T1059.005'],
    # Network / C2
    'connection reset': ['T1071'],
    'connection refused': ['T1071'],
    # DoS / impact
    'outofmemory': ['T1499'],
    'java.lang.outofmemoryerror': ['T1499'],
    # HDFS-Hadoop specific (Remote Services proxy)
    'dataxceiver': ['T1021'],
    'write_block': ['T1021'],
    'read_block': ['T1021'],
}

RANDOM_SEED = 42
MAX_TEXT_CHARS = 1500

def technique_text(tid: str) -> str | None:
    t = tech_by_id.get(tid)
    if t is None:
        return None
    return t['name'] + '. ' + t['description']

def matched_techniques(text: str) -> set[str]:
    lowered = text.lower()
    out: set[str] = set()
    for kw, tids in KEYWORD_MAP.items():
        if kw in lowered:
            for tid in tids:
                if tid in tech_by_id:
                    out.add(tid)
    return out

def build_positive_pairs(
    sample_normal: int | None,
    sample_failure: int | None,
    seed: int = RANDOM_SEED,
) -> list[tuple[str, str]]:
    texts, _, _ = build_windows(
        sample_normal=sample_normal,
        sample_failure=sample_failure,
        random_state=seed,
    )
    rng = random.Random(seed)
    pairs: list[tuple[str, str]] = []
    for text in texts:
        text_trim = text[:MAX_TEXT_CHARS]
        matched = matched_techniques(text)
        if not matched:
            continue
        for tid in sorted(matched):
            tt = technique_text(tid)
            if tt:
                pairs.append((text_trim, tt))
    rng.shuffle(pairs)
    return pairs

keyword_sample_pairs = build_positive_pairs(sample_normal=2000, sample_failure=2000, seed=42)
print(f'keyword-derived sampled positive pairs: {len(keyword_sample_pairs)}')
if keyword_sample_pairs:
    log_t, tech_t = keyword_sample_pairs[0]
    print('\n--- example pair ---')
    print(f'log: {log_t[:300]}{"..." if len(log_t) > 300 else ""}')
    print(f'technique: {tech_t[:300]}{"..." if len(tech_t) > 300 else ""}')

In [ ]:
# OPTIONAL: bring your own (anchor, positive) pairs from a CSV/JSONL on /kaggle/input
# Expected format: CSV with columns ['anchor', 'positive'] OR JSONL with {"anchor": ..., "positive": ...}
# When present, this OVERRIDES the KEYWORD_MAP-derived pairs
import csv

USE_BYO_PAIRS = False
BYO_PAIRS_PATH: str | None = None

def find_byo_pairs_file() -> Path | None:
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for pattern in ('*.csv', '*.jsonl'):
        for path in input_root.rglob(pattern):
            name = path.name.lower()
            if 'pair' in name or 'biencoder' in name:
                return path
    return None

def load_byo_pairs(path: Path) -> list[tuple[str, str]]:
    suffix = path.suffix.lower()
    pairs: list[tuple[str, str]] = []
    if suffix == '.csv':
        with path.open(newline='') as f:
            reader = csv.DictReader(f)
            if not {'anchor', 'positive'}.issubset(reader.fieldnames or []):
                raise ValueError('BYO CSV must include anchor and positive columns')
            for row in reader:
                anchor = (row.get('anchor') or '').strip()
                positive = (row.get('positive') or '').strip()
                if anchor and positive:
                    pairs.append((anchor[:MAX_TEXT_CHARS], positive))
    elif suffix == '.jsonl':
        with path.open() as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                anchor = str(row.get('anchor', '')).strip()
                positive = str(row.get('positive', '')).strip()
                if anchor and positive:
                    pairs.append((anchor[:MAX_TEXT_CHARS], positive))
    else:
        raise ValueError(f'Unsupported BYO pair file: {path}')
    return pairs

def maybe_load_byo_pairs() -> list[tuple[str, str]] | None:
    if not USE_BYO_PAIRS:
        print('BYO pairs disabled; using KEYWORD_MAP-derived pairs.')
        return None
    path = Path(BYO_PAIRS_PATH) if BYO_PAIRS_PATH else find_byo_pairs_file()
    if path is None:
        raise FileNotFoundError('USE_BYO_PAIRS=True but no CSV/JSONL pair file was found under /kaggle/input')
    pairs = load_byo_pairs(path)
    if not pairs:
        raise ValueError(f'No valid BYO pairs loaded from {path}')
    print(f'Loaded BYO pairs: {len(pairs)} from {path}')
    return pairs

byo_pairs = maybe_load_byo_pairs()
sample_pairs = byo_pairs if byo_pairs is not None else keyword_sample_pairs
print(f'active sampled positive pairs: {len(sample_pairs)}')

## 5. Sampled BiEncoder run (verify environment first)

Train two epochs on the 2K-normal + 2K-failure-derived positive pair set with `SentenceTransformer(MODEL_ID='cisco-ai/SecureBERT2.0-biencoder')` and `losses.MultipleNegativesRankingLoss`. Evaluation uses `InformationRetrievalEvaluator`: each log query should retrieve its paired ATT&CK technique text from the eval corpus. Output: `models/biencoder-sample/final/`.

Alternative backbone: if you already ran continued MLM pretraining, replace `MODEL_ID` with `models/securebert2-logs-mlm/final` for serious runs or `models/securebert2-logs-mlm-sample/final` for a quick continued-pretraining check.

In [ ]:
import math
import torch
from sentence_transformers import InputExample, SentenceTransformer, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

MODEL_ID = 'cisco-ai/SecureBERT2.0-biencoder'
EPOCHS = 2
BATCH_SIZE = 32
LR = 2e-5

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA:', torch.cuda.is_available(), 'device:', device, 'bf16:', use_bf16, 'fp16:', use_fp16)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Kaggle run type:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'local-or-interactive'))

if len(sample_pairs) < 2:
    raise ValueError('Need at least two positive pairs for MultipleNegativesRankingLoss.')
split = int(0.9 * len(sample_pairs))
split = min(max(split, 1), len(sample_pairs) - 1)
train_pairs, eval_pairs = sample_pairs[:split], sample_pairs[split:]
train_examples = [InputExample(texts=[anchor, positive]) for anchor, positive in train_pairs]
print(f'train={len(train_examples)}  eval={len(eval_pairs)}')

def make_ir_evaluator(
    pairs: list[tuple[str, str]],
    name: str,
    batch_size: int,
) -> InformationRetrievalEvaluator | None:
    if not pairs:
        return None
    queries: dict[str, str] = {}
    corpus: dict[str, str] = {}
    relevant_docs: dict[str, set[str]] = {}
    corpus_by_text: dict[str, str] = {}
    for i, (anchor, positive) in enumerate(pairs):
        qid = f'q{i}'
        cid = corpus_by_text.setdefault(positive, f't{len(corpus_by_text)}')
        queries[qid] = anchor
        corpus[cid] = positive
        relevant_docs[qid] = {cid}
    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        batch_size=batch_size,
        show_progress_bar=True,
    )

def jsonable_metric(value):
    if isinstance(value, dict):
        return {str(k): jsonable_metric(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable_metric(v) for v in value]
    if hasattr(value, 'item'):
        return value.item()
    if isinstance(value, (int, float, str, bool)) or value is None:
        return value
    return str(value)

model = SentenceTransformer(MODEL_ID, device=device)
train_loss = losses.MultipleNegativesRankingLoss(model)
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
evaluator = make_ir_evaluator(eval_pairs, name='biencoder-logs-attack-sample', batch_size=BATCH_SIZE)
warmup_steps = math.ceil(len(train_loader) * EPOCHS * 0.1)
sample_out = REPO_DIR / 'models' / 'biencoder-sample' / 'final'
sample_out.mkdir(parents=True, exist_ok=True)

model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    use_amp=(use_bf16 or use_fp16),
    show_progress_bar=True,
    output_path=str(sample_out),
    save_best_model=evaluator is not None,
)
if evaluator is None:
    model.save(str(sample_out))

eval_result = evaluator(model, output_path=str(sample_out)) if evaluator is not None else None
metrics = {
    'model_id': MODEL_ID,
    'n_train': len(train_pairs),
    'n_eval': len(eval_pairs),
    'loss': 'MultipleNegativesRankingLoss',
    'evaluator': 'InformationRetrievalEvaluator',
    'eval': jsonable_metric(eval_result),
}
metrics_path = sample_out / 'biencoder_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print('sampled eval metrics:')
print(metrics_path.read_text())
print('saved:', sample_out)

In [ ]:
# OPTIONAL: SimCSE unsupervised path for tighter dedup (cosine 0.95 threshold)
# Uses dropout-based positive generation over raw build_windows text
# No labels required. Useful when no (log, technique) pairs are available.
USE_SIMCSE = False

if USE_SIMCSE:
    import math
    import torch
    from sentence_transformers import InputExample, SentenceTransformer, losses
    from torch.utils.data import DataLoader

    SIMCSE_MODEL_ID = 'cisco-ai/SecureBERT2.0-biencoder'
    simcse_texts, _, _ = build_windows(sample_normal=5000, sample_failure=5000, random_state=42)
    simcse_examples = [
        InputExample(texts=[text[:MAX_TEXT_CHARS], text[:MAX_TEXT_CHARS]])
        for text in simcse_texts
        if text.strip()
    ]
    if len(simcse_examples) < 2:
        raise ValueError('Need at least two raw windows for SimCSE training.')
    simcse_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    simcse_model = SentenceTransformer(SIMCSE_MODEL_ID, device=simcse_device)
    simcse_loss = losses.MultipleNegativesRankingLoss(simcse_model)
    simcse_loader = DataLoader(simcse_examples, shuffle=True, batch_size=32)
    simcse_out = REPO_DIR / 'models' / 'biencoder-simcse-sample' / 'final'
    simcse_out.mkdir(parents=True, exist_ok=True)
    simcse_model.fit(
        train_objectives=[(simcse_loader, simcse_loss)],
        epochs=1,
        warmup_steps=math.ceil(len(simcse_loader) * 0.1),
        optimizer_params={'lr': 2e-5},
        use_amp=torch.cuda.is_available(),
        show_progress_bar=True,
        output_path=str(simcse_out),
    )
    simcse_metrics = {
        'model_id': SIMCSE_MODEL_ID,
        'n_train': len(simcse_examples),
        'loss': 'MultipleNegativesRankingLoss',
        'purpose': 'unsupervised dedup sharpening at cosine 0.95 threshold',
    }
    (simcse_out / 'biencoder_simcse_metrics.json').write_text(json.dumps(simcse_metrics, indent=2))
    print('saved SimCSE artifact:', simcse_out)
else:
    print('SimCSE optional path disabled; set USE_SIMCSE=True to run.')

## 6. Full BiEncoder run (uncomment when sampled run succeeds)

Use the entire HDFS reconstructed corpus to generate keyword-positive pairs. Most normal traces will produce zero positives and be skipped; failure traces carry most of the pair volume. Output: `models/biencoder/final/`. Estimated runtime is roughly 1-2h on Kaggle T4, depending on pair count and evaluator corpus size.

In [ ]:
# full_pairs = maybe_load_byo_pairs()
# if full_pairs is None:
#     full_pairs = build_positive_pairs(sample_normal=None, sample_failure=None, seed=42)
# print(f'full positive pairs: {len(full_pairs)}')
# if len(full_pairs) < 2:
#     raise ValueError('Need at least two positive pairs for full BiEncoder training.')
#
# split = int(0.9 * len(full_pairs))
# split = min(max(split, 1), len(full_pairs) - 1)
# train_pairs_f, eval_pairs_f = full_pairs[:split], full_pairs[split:]
# train_examples_f = [InputExample(texts=[anchor, positive]) for anchor, positive in train_pairs_f]
# train_loader_f = DataLoader(train_examples_f, shuffle=True, batch_size=BATCH_SIZE)
# evaluator_f = make_ir_evaluator(eval_pairs_f, name='biencoder-logs-attack-full', batch_size=BATCH_SIZE)
# warmup_f = math.ceil(len(train_loader_f) * EPOCHS * 0.1)
# full_out = REPO_DIR / 'models' / 'biencoder' / 'final'
# full_out.mkdir(parents=True, exist_ok=True)
#
# model_f = SentenceTransformer(MODEL_ID, device=device)
# train_loss_f = losses.MultipleNegativesRankingLoss(model_f)
# model_f.fit(
#     train_objectives=[(train_loader_f, train_loss_f)],
#     evaluator=evaluator_f,
#     epochs=EPOCHS,
#     warmup_steps=warmup_f,
#     optimizer_params={'lr': LR},
#     use_amp=(use_bf16 or use_fp16),
#     show_progress_bar=True,
#     output_path=str(full_out),
#     save_best_model=evaluator_f is not None,
# )
# if evaluator_f is None:
#     model_f.save(str(full_out))
# eval_result_f = evaluator_f(model_f, output_path=str(full_out)) if evaluator_f is not None else None
# metrics_f = {
#     'model_id': MODEL_ID,
#     'n_train': len(train_pairs_f),
#     'n_eval': len(eval_pairs_f),
#     'loss': 'MultipleNegativesRankingLoss',
#     'evaluator': 'InformationRetrievalEvaluator',
#     'eval': jsonable_metric(eval_result_f),
# }
# (full_out / 'biencoder_metrics.json').write_text(json.dumps(metrics_f, indent=2))
# print('full eval:', metrics_f)

## 7. Inspect artifacts

Confirm the output directory has the standard sentence-transformers BiEncoder layout. The inspection highlights the model card, `sentence_bert_config.json`, tokenizer files, and `biencoder_metrics.json` eval summary.

In [ ]:
interesting_files = [
    ('model_card', 'README.md'),
    ('model_card', 'model_card.md'),
    ('sentence_bert_config', 'sentence_bert_config.json'),
    ('sentence_transformers_config', 'config_sentence_transformers.json'),
    ('tokenizer', 'tokenizer.json'),
    ('tokenizer_config', 'tokenizer_config.json'),
    ('special_tokens', 'special_tokens_map.json'),
    ('eval_metrics', 'biencoder_metrics.json'),
    ('simcse_metrics', 'biencoder_simcse_metrics.json'),
]

for candidate in [
    REPO_DIR / 'models' / 'biencoder' / 'final',
    REPO_DIR / 'models' / 'biencoder-sample' / 'final',
    REPO_DIR / 'models' / 'biencoder-simcse-sample' / 'final',
]:
    if not candidate.exists():
        continue
    print(f'=== {candidate} ===')
    for path in sorted(candidate.rglob('*')):
        if path.is_file():
            print(f'  {path.relative_to(candidate)}: {path.stat().st_size:,} bytes')
    for label, filename in interesting_files:
        path = candidate / filename
        if not path.exists():
            continue
        print(f'\n--- {label}: {filename} ---')
        text = path.read_text(errors='replace')
        if filename.endswith('.json'):
            print(json.dumps(json.loads(text), indent=2)[:2000])
        else:
            print(text[:2000])

## 8. Package artifacts

Zip the selected BiEncoder directory into `/kaggle/working/biencoder-final.zip`. The full artifact is preferred when present; otherwise the sampled artifact is packaged for verification download.

In [ ]:
import shutil

full_dir = REPO_DIR / 'models' / 'biencoder' / 'final'
sample_dir = REPO_DIR / 'models' / 'biencoder-sample' / 'final'
tag = 'full' if full_dir.exists() else 'sample'
src_dir = full_dir if tag == 'full' else sample_dir
if not src_dir.exists():
    raise FileNotFoundError('No BiEncoder artifact found. Run the sampled or full training cell first.')

out_zip = KAGGLE_WORKING / 'biencoder-final'
zip_path = out_zip.with_suffix('.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(
    str(out_zip),
    'zip',
    root_dir=REPO_DIR,
    base_dir=Path(*src_dir.parts[-3:]).as_posix(),
)
print('packaged:', zip_path)
print('artifact tag:', tag, 'source:', src_dir)

## 9. Output description + how to consume in repo

Download `/kaggle/working/biencoder-final.zip` and extract it at the repository root. A full run archive contains:

```text
models/biencoder/final/modules.json
models/biencoder/final/config_sentence_transformers.json
models/biencoder/final/sentence_bert_config.json
models/biencoder/final/model.safetensors
models/biencoder/final/tokenizer.json (and tokenizer_config.json, special_tokens_map.json)
models/biencoder/final/biencoder_metrics.json
```

To use this fine-tuned BiEncoder inside the production pipeline, make this exact one-line config change:

```yaml
models.biencoder.model_id: "models/biencoder/final"
```

Equivalent block form in `config/config.yaml`:

```yaml
models:
  biencoder:
    model_id: "models/biencoder/final"   # was: cisco-ai/SecureBERT2.0-biencoder
    device: "cpu"
    batch_size: 64
    dedup_threshold: 0.95
    dedup_window_minutes: 5
    faiss_top_k: 3
```

Caveats:

* **Bootstrap labels are heuristic.** `biencoder_metrics.json` measures agreement with `KEYWORD_MAP`, not analyst-verified semantic correctness. Treat it as a relative signal between runs.
* **Retrieval calibration can shift.** Re-check top-k ATT&CK retrieval quality and the `dedup_threshold: 0.95` operating point on representative logs before deployment. The downstream CrossEncoder will only see candidates retrieved here.
* **Composability with MLM pre-training.** For best results, run `kaggle_pretrain_mlm.ipynb` first and switch `MODEL_ID` to `models/securebert2-logs-mlm/final`. The `models/securebert2-logs-mlm-sample/final` artifact is a lightweight alternative starting backbone for quick checks.
* **No labelled pairs available?** Use the optional SimCSE cell to sharpen dedup embeddings from raw log windows, then return to labelled log-to-technique pairs when you have them.